<a href="https://colab.research.google.com/github/hamidatbi/himalaya-glof-risk/blob/main/Hymalias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.7 MB/s  0:00:00


In [2]:
import ee
import geemap
print("all imported")

all imported


In [3]:
ee.Authenticate()
ee.Initialize(project='himalayan-glacial-risk')
print("done")

done


In [4]:
lake=ee.Geometry.Point([88.198, 27.913])
print('lake saved')

lake saved


In [5]:
s2=ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
print("Saved")

Saved


In [7]:
before= (s2
         .filterBounds(lake)
         .filterDate('2022-10-10', '2022-11-30')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)))

after= (s2
         .filterBounds(lake)
         .filterDate('2023-10-10', '2023-11-30')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)))

print('Before images:', before.size().getInfo())
print('After images:', after.size().getInfo())

Before images: 16
After images: 14


In [8]:
before_img = before.median()
after_img = after.median()
print('done')

done


In [9]:
vis= {'bands': ['B4', 'B3', 'B2'],
      'min': 0,
      'max': 6000}

map=geemap.Map(center=[27.913, 88.198], zoom=14, basemap='Esri.WorldImagery')

map.addLayer(before_img, vis, 'In 2022')
map.addLayer(after_img, vis, 'In 2023')

map

Map(center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [10]:
admin_bounds=ee.FeatureCollection("FAO/GAUL/2025/level1")
study_area = admin_bounds.filter(
    ee.Filter.Or(
        ee.Filter.inList('GAUL0_NAME', ['Bhutan', 'Nepal']),
        ee.Filter.eq('GAUL1_NAME', 'Sikkim')
    )
)

study_area.size().getInfo()

28

to see the columns:  geemap.ee_to_df(admin_bounds.limit(5))

In [ ]:
#geemap.ee_to_df(admin_bounds.limit(5))

import pandas as pd

india = admin_bounds.filter(ee.Filter.eq('GAUL0_NAME', 'India'))
df = geemap.ee_to_df(india)

#df.head()
df[df['GAUL1_NAME']=='Sikkim']

,CONTINENT,DISP_EN,FID,GAUL0_CODE,GAUL0_NAME,GAUL1_CODE,GAUL1_NAME,ISO3_CODE,MAP_CODE
24,Asia,"Sikkim, India",1325,240,India,2608,Sikkim,IND,IND


In [11]:
map2=geemap.Map(center=[28, 86.5], zoom=6, basemap='Esri.WorldImagery')
map2.addLayer(study_area, {}, 'Study area')

aoi=study_area.union()
map2.addLayer(aoi, {'color': 'red'}, 'AOI')

map2

Map(center=[28, 86.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [12]:
copernicus=ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1")
dem=copernicus.select('DEM')
mosaic=dem.mosaic()
dem_aoi=mosaic.clip(aoi.geometry())


vis= {'palette':['green', 'yellow', 'brown', 'white'],
      'min': 0,
      'max': 8000}

map=geemap.Map(center=[27.913, 88.198], zoom=7, basemap='Esri.WorldImagery')
map.addLayer(dem_aoi, vis, 'DEM of study area')

map

Map(center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [14]:
high_elev=dem_aoi.gt(4000)
masked_hel=dem_aoi.updateMask(high_elev)

map.addLayer(masked_hel,vis,'Elevations above 4k meters')
map

Map(bottom=14036.0, center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], p…